# 01 — Data Collection Pipeline

Dự án này sử dụng dữ liệu tích hợp từ hai nguồn chính thông qua giao diện lập trình ứng dụng (APIs) công khai:
1. **SteamSpy API**: Cung cấp các thông tin thống kê về lượng đánh giá (positive/negative reviews), lượng người chơi đồng thời (CCU) và lượng người sở hữu ước lượng (owners range).
2. **Steam Store API**: Cung cấp thông tin chi tiết về game bao gồm: tên game chính thức, nhà phát triển (developer), nhà xuất bản (publisher), giá bán gốc (price), trạng thái miễn phí (is_free), và danh sách thể loại (genres).

Notebook này tài liệu hóa quy trình thu thập dữ liệu tự động, giải quyết vấn đề Entity Resolution và chuẩn bị tệp thô gốc tại `data/raw/steam_games_raw.csv`.

In [ ]:
import logging
import os
import time
from datetime import datetime
from typing import Any
import pandas as pd
import requests
import steamspypi

# Cấu hình tham số thu thập
MAX_PAGES = 5            # Chạy thử 5 trang đầu tiên (Demo)
REQUEST_DELAY = 1.0      # Độ trễ giữa các lượt gọi API để tránh bị chặn (Rate Limit)
MIN_APPID = 1500000      # Chỉ lấy các game từ khoảng 2022 trở đi để tối ưu thời gian crawl
OUTPUT_PATH = "../data/raw/steam_games_raw.csv"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

### 1. Thu thập danh sách trò chơi từ SteamSpy API

Chúng ta gọi API SteamSpy để lấy danh sách các game phổ biến và các thông số thống kê đi kèm.

In [ ]:
def fetch_steamspy_appids(pages: int = MAX_PAGES) -> pd.DataFrame:
    all_data = []
    for page in range(pages):
        try: 
            print(f"Fetching SteamSpy page {page}...")
            data = steamspypi.download({"request": "all", "page": page})
            if not data:
                break
            df = pd.DataFrame(data).T.reset_index()
            df = df.rename(columns={"index": "appid"})
            cols = df.columns.tolist()
            dup_count = cols.count("appid")
            if dup_count > 1:
                first = cols.index("appid")
                df = df.iloc[:, : first + 1].join(df.iloc[:, first + 2 :])
            all_data.append(df)
        except Exception as exc:
            print(f"Error on page {page}: {exc}")
            break
        time.sleep(0.5)
    
    if not all_data:
        return pd.DataFrame()

    combined = pd.concat(all_data, ignore_index=True)
    combined = combined.drop_duplicates(subset="appid", keep="first")
    print(f"Total unique games from SteamSpy: {len(combined)}")
    return combined

### 2. Làm giàu dữ liệu bằng Steam Store API (Enrichment & Entity Resolution)

Steam Store API cung cấp giá bán thực tế, ngày phát hành và các thể loại chính thức của game. Ở bước này chúng ta xử lý:
- **Price Bug Fix**: Giá bán trả về từ Store API là dạng cents (ví dụ: 1499 đại diện cho $14.99). Chúng ta chia cho 100 để lưu trữ dạng USD.
- **Entity Resolution**: Đồng bộ tên game, nhà phát triển và nhà xuất bản từ cả hai API để có bộ thuộc tính hợp nhất duy nhất.

In [ ]:
def fetch_store_details(appid: int) -> dict | None:
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            payload = resp.json()
            app_key = str(appid)
            if app_key in payload and payload[app_key].get("success"):
                return payload[app_key]["data"]
    except Exception as exc:
        pass
    return None

def enrich_with_store_data(steamspy_df: pd.DataFrame) -> pd.DataFrame:
    # Lấy 10 games mẫu để minh họa chạy thử nhanh
    appids = steamspy_df["appid"].unique().tolist()[:10]
    print(f"Enriching {len(appids)} games with Steam Store data...")
    
    records = []
    for i, appid in enumerate(appids, 1):
        details = fetch_store_details(appid)
        if details:
            price_overview = details.get("price_overview") or {}
            store_price_cents = price_overview.get("final")
            records.append({
                "appid": appid,
                "store_name": details.get("name"),
                "release_date": details.get("release_date", {}).get("date"),
                "genres": [g["description"] for g in details.get("genres", [])],
                "developer": (details.get("developers") or [None])[0],
                "publisher": (details.get("publishers") or [None])[0],
                "store_price": store_price_cents / 100.0 if store_price_cents is not None else None,
                "is_free": details.get("is_free", False),
            })
        time.sleep(REQUEST_DELAY)
        
    store_df = pd.DataFrame(records)
    store_df["appid"] = store_df["appid"].astype("int64")
    steamspy_df["appid"] = steamspy_df["appid"].astype("int64")
    
    merged = pd.merge(steamspy_df, store_df, on="appid", how="inner")
    
    # Trộn giá
    spy_price = pd.to_numeric(merged["price"], errors="coerce")
    store_price = merged["store_price"]
    merged["price"] = store_price.where(store_price.notna(), spy_price)
    merged["price"] = merged["price"].fillna(0)
    merged = merged.drop(columns=["store_price"], errors="ignore")
    return merged

### 3. Chuẩn hóa ngày phát hành và lọc theo năm nghiên cứu (2022-2026)

Hàm phân tích chuỗi ngày sang định dạng datetime chuẩn và gán cột năm phát hành.

In [ ]:
_DATE_FORMATS = ["%d %b, %Y", "%b %d, %Y", "%Y-%m-%d"]

def parse_release_date(date_str: str | None) -> datetime | None:
    if not date_str or pd.isna(date_str):
        return None
    for fmt in _DATE_FORMATS:
        try:
            return datetime.strptime(date_str.strip(), fmt)
        except ValueError:
            continue
    return None

### 4. Khởi động quy trình thu thập dữ liệu thử nghiệm

Đoạn code sau đây minh họa cách thức chạy quy trình thu thập dữ liệu thu nhỏ.

In [ ]:
# Chạy thử nghiệm quy trình
try:
    spy_df = fetch_steamspy_appids(pages=1)
    if not spy_df.empty:
        # Lọc thô theo appid tối thiểu để demo nhanh
        spy_df["appid"] = pd.to_numeric(spy_df["appid"], errors="coerce")
        filtered_spy = spy_df[spy_df["appid"] >= MIN_APPID].copy()
        print(f"Games matching filter: {len(filtered_spy)}")
        if len(filtered_spy) > 0:
            final_df = enrich_with_store_data(filtered_spy)
            final_df["release_date"] = final_df["release_date"].apply(parse_release_date)
            final_df["year"] = final_df["release_date"].dt.year.astype("Int64")
            print(f"Sample collection complete. Shape: {final_df.shape}")
            print(final_df.head(2))
except Exception as e:
    print(f"Demo data collection interrupted: {e}")